# 06 - Multi-seed + Confidence Interval (TANPA SFM)

**Tujuan:** menghasilkan angka baseline yang LAYAK MASUK PAPER -- bukan sekadar satu
angka ilustratif, tapi rata-rata +/- 95% CI lintas beberapa seed. Ini menutup jarak
antara jalur belajar (nb02-05) dan tabel hasil `../paper-survey.md` (R1, R2).

Konsisten dengan `../02_survey-tanpaSFM/s07_multiseed_ci.ipynb`: seed {13,42,101,202,303}.

**Yang diukur:**
- **R1** FedAvg vs FedProx (pada partisi non-IID) -- MCC mean +/- CI.
- **R2** sensitivitas non-IID -- MCC vs Dirichlet alpha, mean +/- CI.

**Catatan kejujuran:** angka = hasil eksekusi nyata di SageMaker. CI dari variasi seed
(keacakan init + mini-batch + partisi), dilaporkan apa adanya. Single-dataset
(CIC-IDS2017) -> ini baseline fondasi; versi cross-dataset ada di survey (s03).

**Prasyarat:** `01_data_partition.ipynb`. Dijalankan di SageMaker.


In [ ]:
import sys
!{sys.executable} -m pip install -q numpy scikit-learn scipy
print('deps OK')

## 1. MLP + FedAvg/FedProx (identik fed_node_survey.py)


In [ ]:
import numpy as np
def init_weights(d, seed):
    r=np.random.RandomState(seed); L=[d,128,64,1]; W=[]
    for a,b in zip(L[:-1],L[1:]):
        W.append(r.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def _relu(x): return np.maximum(0,x)
def _sig(x):  return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))
def forward(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1)
    a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]; return _sig(a3).ravel(),(X,a1,z1,a2,z2,a3)
def train_local(W,X,y,epochs=2,lr=3e-3,batch=256,mu=0.0,W_global=None,seed=42):
    r=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=r.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            o,(X_,a1,z1,a2,z2,a3)=forward(W,Xb)
            g=(o-yb)/len(yb)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0); gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0);        gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                gv=np.asarray(gv).reshape(W[i].shape)
                if mu>0 and W_global is not None: gv=gv+mu*(W[i]-W_global[i])
                W[i]=W[i]-lr*gv
    return W
def fedavg(ws,sz):
    tot=float(sum(sz)); return [sum((sz[k]/tot)*ws[k][i] for k in range(len(ws))) for i in range(len(ws[0]))]
def predict(W,X): return forward(W,X)[0]
def zfit(X): m=X.mean(0); s=X.std(0); s[s==0]=1.0; return m,s
def zapp(X,m,s): return (X-m)/s
print('ok')

## 2. Data + partisi Dirichlet (seed-dependent)


In [ ]:
import os
DATA='./fl_data'
assert os.path.isdir(DATA), 'Jalankan 01_data_partition.ipynb dulu'
cs=sorted(set(int(f.split('_')[1]) for f in os.listdir(DATA) if f.startswith('client_')))
Xpop=np.vstack([np.load(f'{DATA}/client_{i}_train.npz')['X'] for i in cs])
ypop=np.concatenate([np.load(f'{DATA}/client_{i}_train.npz')['y'] for i in cs])
g=np.load(f'{DATA}/global_test.npz'); Xg,yg=g['X'],g['y']; d=Xg.shape[1]; K=4
def dirichlet_partition(y,K,alpha,seed):
    r=np.random.RandomState(seed); out=[[] for _ in range(K)]
    for c in np.unique(y):
        ci=np.where(y==c)[0]; r.shuffle(ci)
        cuts=(np.cumsum(r.dirichlet([alpha]*K))*len(ci)).astype(int)[:-1]
        for k,part in enumerate(np.split(ci,cuts)): out[k]+=part.tolist()
    return [np.array(sorted(ix)) for ix in out]
print('populasi:', Xpop.shape, '| d =', d)

## 3. Satu run federasi (parametrik: alpha, mu, seed)


In [ ]:
from sklearn.metrics import matthews_corrcoef
def one_run(alpha, mu, seed, rounds=30, local_epochs=2):
    parts=dirichlet_partition(ypop,K,alpha,seed)
    cli=[]
    for ix in parts:
        if len(ix)<10: continue
        Xi,yi=Xpop[ix],ypop[ix]; m_,s_=zfit(Xi)
        cli.append(dict(Xz=zapp(Xi,m_,s_), y=yi, mu=m_, sd=s_, n=len(yi)))
    W=init_weights(d,seed)
    for t in range(rounds):
        cw=[train_local(W,c['Xz'],c['y'],epochs=local_epochs,mu=mu,W_global=W,seed=seed) for c in cli]
        W=fedavg(cw,[c['n'] for c in cli])
    ms=[matthews_corrcoef(yg,(predict(W,zapp(Xg,c['mu'],c['sd']))>=0.5).astype(int)) for c in cli]
    return float(np.mean(ms))

## 4. Helper CI (95%, t-interval)


In [ ]:
from scipy import stats
def mean_ci(vals, conf=0.95):
    a=np.array(vals,float); n=len(a); m=a.mean()
    if n<2: return m,0.0
    se=a.std(ddof=1)/np.sqrt(n); h=se*stats.t.ppf((1+conf)/2.0, n-1)
    return float(m), float(h)
SEEDS=[13,42,101,202,303]
print('seeds:', SEEDS)

## 5. R1 -- FedAvg vs FedProx (non-IID alpha=0.1), multi-seed


In [ ]:
ALPHA_R1=0.1
r1={}
for name,mu in [('FedAvg',0.0),('FedProx(mu=0.01)',0.01),('FedProx(mu=0.1)',0.1)]:
    vals=[one_run(ALPHA_R1, mu, sd) for sd in SEEDS]
    m,h=mean_ci(vals); r1[name]=(m,h,vals)
    print(f'{name:18s}: MCC = {m:.4f} +/- {h:.4f}  (seeds: {[round(v,3) for v in vals]})')

## 6. R2 -- sensitivitas non-IID (sweep alpha), multi-seed FedAvg


In [ ]:
ALPHAS=[100.0,1.0,0.5,0.1]
r2={}
for a in ALPHAS:
    vals=[one_run(a, 0.0, sd) for sd in SEEDS]
    m,h=mean_ci(vals); r2[a]=(m,h,vals)
    print(f'alpha={a:>6}: MCC = {m:.4f} +/- {h:.4f}')

## 7. Simpan tabel siap-paper

CSV ini formatnya cocok untuk mengisi R1 & R2 di `../paper-survey.md` Bagian 4.3
(kolom 'Angka lab kami'). Pakai apa adanya; jangan diedit manual angkanya.


In [ ]:
import csv
os.makedirs('learn_out',exist_ok=True)
with open('learn_out/06_multiseed_ci.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['claim','setting','method','mcc_mean','ci95_halfwidth','n_seeds'])
    for name,(m,h,vals) in r1.items():
        w.writerow(['R1',f'noniid_alpha={ALPHA_R1}',name,round(m,4),round(h,4),len(vals)])
    for a,(m,h,vals) in r2.items():
        w.writerow(['R2',f'alpha={a}','FedAvg',round(m,4),round(h,4),len(vals)])
print('tersimpan: learn_out/06_multiseed_ci.csv')
# ringkas arah temuan (untuk narasi paper, bukan angka hard-coded)
fa=r1['FedAvg'][0]; fp=max(r1['FedProx(mu=0.01)'][0], r1['FedProx(mu=0.1)'][0])
print(f'\nR1 arah: FedProx {"<=" if fp<=fa else ">"} FedAvg  (FedAvg {fa:.4f} vs FedProx-terbaik {fp:.4f})')
print(f'R2 arah: alpha=100 -> {r2[100.0][0]:.4f} ; alpha=0.1 -> {r2[0.1][0]:.4f} (gap {r2[100.0][0]-r2[0.1][0]:+.4f})')
print('\nIsi R1/R2 di paper-survey.md Bagian 4.3 dari CSV ini. Cross-dataset (R3) -> lihat survey/s03.')

In [ ]:
# === UPLOAD learn_out/ -> S3 (arsip hasil) ===
# Jalankan setelah semua CSV 02-06 selesai. Mengunggah seluruh folder
# learn_out/ ke S3 agar hasil terpusat & bisa di-download ke laptop untuk
# mengisi paper-survey.md. Angka tidak diubah; ini murni sinkronisasi file.
import subprocess, os

# --- KONFIG upload (samakan bucket dengan 01_data_partition.ipynb) ---
S3_BUCKET = 'ssh-detection-features-232032302717'
S3_DEST   = 's3://%s/federated/learn_out_tanpaSFM/' % S3_BUCKET
LOCAL_DIR = './learn_out'

assert os.path.isdir(LOCAL_DIR), 'learn_out/ belum ada -- jalankan nb 02-06 dulu.'
csvs = [f for f in os.listdir(LOCAL_DIR) if f.endswith('.csv')]
print('CSV yang akan diunggah:', sorted(csvs))

cmd = ['aws', 's3', 'sync', LOCAL_DIR, S3_DEST]
print('>>', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print('GAGAL upload (cek kredensial/izin IAM instance):')
    print(res.stderr)
else:
    print('Selesai. Hasil tersimpan di', S3_DEST)
    print('Download ke laptop:  aws s3 sync %s ./learn_out' % S3_DEST)
